# Truy xuất và Lưu trữ Dữ liệu với Python — Notebook Thực hành

**Ngôn ngữ:** Python 3  
**Thời lượng gợi ý:** 150–180 phút  
**Bối cảnh:** Phân tích kinh doanh, khách hàng, sản phẩm, bán hàng và vận hành  
**Nguồn dữ liệu:** CSV, Excel, JSON, HTML, PDF, SQLite và dữ liệu dạng document của MongoDB

Notebook này đi cùng bài giảng **Truy xuất và Lưu trữ Dữ liệu với Python**.

Mỗi phần được tổ chức theo cùng một trình tự học tập:

1. Khái niệm hoặc câu lệnh cốt lõi;
2. Ví dụ hoàn chỉnh có thể chạy trực tiếp;
3. Bài tập được đặt ngay sau nội dung liên quan;
4. Gợi ý và mã khung `TODO` chưa hoàn thiện;
5. Bài thực hành tích hợp ở phần cuối.

> Cách học đề xuất: chạy notebook từ trên xuống một lần để quan sát toàn bộ quy trình; sau đó quay lại các ô có nhãn `TODO`, hoàn thiện mã và chạy lại từng ô.

## Mục tiêu học tập

Sau khi hoàn thành notebook này, người học có thể:

- đọc và ghi dữ liệu số dạng văn bản bằng NumPy;
- nhập dữ liệu CSV bằng Pandas với `usecols`, `dtype`, `parse_dates`, `na_values` và `chunksize`;
- đọc và ghi workbook Excel có nhiều trang tính;
- đọc JSON và làm phẳng cấu trúc JSON lồng nhau;
- trích xuất bảng từ HTML và hiểu quy trình cơ bản để trích xuất bảng từ PDF;
- kết nối SQLite, thực hiện truy vấn, sử dụng tham số và chuyển dữ liệu giữa SQL và Pandas;
- làm việc với dữ liệu dạng document tương tự MongoDB và chuyển sang DataFrame;
- tích hợp nhiều nguồn thành một tập dữ liệu phân tích thống nhất;
- lưu kết quả dưới dạng CSV, Excel và SQLite.

## 0. Chuẩn bị môi trường

Nếu môi trường chưa có các thư viện cần thiết, có thể cài đặt bằng:

```bash
pip install numpy pandas openpyxl lxml html5lib beautifulsoup4 pdfplumber reportlab pymongo
```

Notebook tự tạo các tệp dữ liệu mẫu nên không cần tải thêm dataset bên ngoài.

In [ ]:
import sys
import json
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

BASE_DIR = Path("data_access_storage_lab")
SOURCE_DIR = BASE_DIR / "sources"
OUTPUT_DIR = BASE_DIR / "outputs"

SOURCE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Thư mục làm việc:", BASE_DIR.resolve())

## 1. Tạo các nguồn dữ liệu mẫu

Notebook sử dụng một bối cảnh bán lẻ thống nhất xuyên suốt các phần thực hành.

Ô thiết lập sẽ tạo:

- `customers.csv`;
- `numeric_sales.csv`;
- `products.xlsx`;
- `orders.json`;
- `market_data.html`;
- `company.db`;
- `large_transactions.csv`.

Một báo cáo PDF nhỏ cũng được tạo nếu môi trường đã cài `reportlab`.

Phần tạo nguồn dữ liệu đã được viết hoàn chỉnh để người học tập trung vào **truy xuất, kiểm tra, tích hợp và lưu trữ dữ liệu**, thay vì dành thời gian xây dựng dữ liệu thô.

In [ ]:
rng = np.random.default_rng(42)

# customers.csv — dữ liệu khách hàng
customers = pd.DataFrame({
    "CustomerID": ["001", "002", "003", "004", "005", "006", "007", "008"],
    "CustomerName": [
        "An Nguyen", "Binh Tran", "Chi Le", "Dung Pham",
        "Giang Vu", "Ha Do", "Khanh Hoang", "Linh Bui"
    ],
    "City": ["Hanoi", "Hanoi", "Danang", "HCMC", "HCMC", "Danang", "Hanoi", "HCMC"],
    "Age": [28, 35, 31, 42, np.nan, 26, 39, 33],
    "SignupDate": [
        "2025-01-10", "2025-02-18", "2025-03-07", "2025-03-21",
        "2025-04-15", "2025-05-09", "2025-06-01", "2025-06-19"
    ],
    "Segment": ["Retail", "Corporate", "Retail", "SME", "Corporate", "Retail", "SME", "Corporate"],
    "TotalSpent": [1250, 2480, 890, 3150, 2760, 720, 1980, 3520]
})

customers_file = customers.copy().astype({"CustomerID": str})
customers_file["Age"] = customers_file["Age"].astype(object)
customers_file.loc[4, "Age"] = "NA"
customers_file.loc[5, "Segment"] = "-"
customers_file.to_csv(SOURCE_DIR / "customers.csv", index=False)

# numeric_sales.csv — dữ liệu bán hàng dạng số
numeric_sales = pd.DataFrame({
    "Month": [1, 2, 3, 4, 5, 6],
    "Revenue": [120, 150, 180, 210, 240, 260],
    "Cost": [80, np.nan, 110, 135, np.nan, 170]
})
numeric_sales.to_csv(SOURCE_DIR / "numeric_sales.csv", index=False)

# products.xlsx — sản phẩm và tồn kho
products = pd.DataFrame({
    "ProductID": ["P01", "P02", "P03", "P04", "P05"],
    "ProductName": ["Laptop", "Monitor", "Keyboard", "Mouse", "Headset"],
    "Category": ["Computers", "Accessories", "Accessories", "Accessories", "Accessories"],
    "UnitPrice": [1200, 320, 80, 35, 95]
})

inventory = pd.DataFrame({
    "ProductID": ["P01", "P02", "P03", "P04", "P05"],
    "Stock": [12, 25, 60, 90, 45],
    "ReorderLevel": [5, 10, 20, 30, 15]
})

with pd.ExcelWriter(SOURCE_DIR / "products.xlsx", engine="openpyxl") as writer:
    products.to_excel(writer, sheet_name="Products", index=False)
    inventory.to_excel(writer, sheet_name="Inventory", index=False)

# orders.json — đơn hàng lồng nhau
orders = [
    {
        "OrderID": "O001",
        "OrderDate": "2025-07-01",
        "Customer": {"CustomerID": "001", "City": "Hanoi"},
        "Items": [
            {"ProductID": "P01", "Quantity": 1},
            {"ProductID": "P04", "Quantity": 2}
        ]
    },
    {
        "OrderID": "O002",
        "OrderDate": "2025-07-02",
        "Customer": {"CustomerID": "003", "City": "Danang"},
        "Items": [
            {"ProductID": "P02", "Quantity": 2},
            {"ProductID": "P03", "Quantity": 3}
        ]
    },
    {
        "OrderID": "O003",
        "OrderDate": "2025-07-03",
        "Customer": {"CustomerID": "008", "City": "HCMC"},
        "Items": [
            {"ProductID": "P05", "Quantity": 2},
            {"ProductID": "P04", "Quantity": 1}
        ]
    },
    {
        "OrderID": "O004",
        "OrderDate": "2025-07-04",
        "Customer": {"CustomerID": "006", "City": "Danang"},
        "Items": [
            {"ProductID": "P03", "Quantity": 1},
            {"ProductID": "P04", "Quantity": 2}
        ]
    }
]

with open(SOURCE_DIR / "orders.json", "w", encoding="utf-8") as f:
    json.dump(orders, f, ensure_ascii=False, indent=2)

# market_data.html — dữ liệu thị trường
market_prices = pd.DataFrame({
    "Product": ["Laptop", "Monitor", "Keyboard"],
    "MarketPrice": [1225, 335, 82]
})
fx_rates = pd.DataFrame({
    "Currency": ["USD", "EUR", "JPY"],
    "RateToVND": [25200, 29600, 171]
})

html = (
    "<html><body><h1>Market Data</h1>"
    + market_prices.to_html(index=False)
    + "<h2>FX Rates</h2>"
    + fx_rates.to_html(index=False)
    + "</body></html>"
)
(SOURCE_DIR / "market_data.html").write_text(html, encoding="utf-8")

# company.db — cơ sở dữ liệu SQLite
conn_setup = sqlite3.connect(SOURCE_DIR / "company.db")
employees = pd.DataFrame({
    "EmployeeID": [1, 2, 3, 4, 5],
    "EmployeeName": ["Mai", "Nam", "Hoa", "Son", "Trang"],
    "Department": ["Sales", "Sales", "Operations", "Finance", "Operations"],
    "Salary": [28000000, 34000000, 30000000, 42000000, 31500000]
})
employees.to_sql("employees", conn_setup, if_exists="replace", index=False)
customers[["CustomerID", "CustomerName", "City"]].to_sql(
    "customers", conn_setup, if_exists="replace", index=False
)
conn_setup.close()

# large_transactions.csv — dữ liệu giao dịch lớn
n_rows = 20000
large_transactions = pd.DataFrame({
    "TransactionID": np.arange(1, n_rows + 1),
    "CustomerID": rng.choice(customers["CustomerID"], n_rows),
    "Amount": rng.gamma(shape=2.5, scale=80, size=n_rows).round(2),
    "Channel": rng.choice(["Online", "Store", "Partner"], n_rows, p=[0.55, 0.30, 0.15])
})
large_transactions.to_csv(SOURCE_DIR / "large_transactions.csv", index=False)

print("Các tệp nguồn đã được tạo:")
for path in sorted(SOURCE_DIR.iterdir()):
    print("-", path.name)

### Kiểm tra nguồn dữ liệu trước khi phân tích

Trước khi sử dụng bất kỳ nguồn dữ liệu nào, hãy xác định:

1. mỗi hàng hoặc mỗi document đại diện cho đối tượng nào;
2. trường nào là mã định danh, biến phân loại, biến số hoặc ngày tháng;
3. dữ liệu có sử dụng ký hiệu đặc biệt nào để biểu diễn giá trị thiếu hay không;
4. trường nào có thể được sử dụng làm khóa khi tích hợp dữ liệu.

In [ ]:
preview = pd.read_csv(SOURCE_DIR / "customers.csv")
print(preview.head().to_string(index=False))
print("\nShape:", preview.shape)
print("\nData types:")
print(preview.dtypes)

# Phần 1. Dữ liệu CSV dạng số với NumPy

## 2. Đọc dữ liệu số đơn giản bằng `np.loadtxt()`

`np.loadtxt()` phù hợp với dữ liệu số có cấu trúc đều và không có các giá trị thiếu gây vấn đề khi đọc.

In [ ]:
sales_numeric = np.loadtxt(
    SOURCE_DIR / "numeric_sales.csv",
    delimiter=",",
    skiprows=1,
    usecols=[0, 1]
)

print(sales_numeric)
print("Kích thước:", sales_numeric.shape)
print("Doanh thu trung bình:", sales_numeric[:, 1].mean())

### Bài tập 1 — `np.loadtxt()`

Đọc hai cột `Month` và `Revenue` từ `numeric_sales.csv`.

Sau đó:

1. in mảng dữ liệu;
2. in kích thước của mảng;
3. tính doanh thu lớn nhất.

**Gợi ý:** sử dụng `delimiter=","`, `skiprows=1` và `usecols=[0, 1]`.

In [ ]:
# TODO
# arr_ex1 = np.loadtxt(
#     SOURCE_DIR / "numeric_sales.csv",
#     delimiter=...,
#     skiprows=...,
#     usecols=...
# )
# print(arr_ex1)
# print(arr_ex1.shape)
# print(arr_ex1[:, 1].max())

## 3. Xử lý giá trị thiếu bằng `np.genfromtxt()`

`np.genfromtxt()` linh hoạt hơn khi tệp dữ liệu số có chứa giá trị thiếu.

In [ ]:
sales_with_missing = np.genfromtxt(
    SOURCE_DIR / "numeric_sales.csv",
    delimiter=",",
    skip_header=1,
    filling_values=0.0
)

print(sales_with_missing)

### Bài tập 2 — `np.genfromtxt()`

Đọc lại tệp trên và thay mọi giá trị thiếu bằng `-1`.

Sau đó kiểm tra cột `Cost`.

**Gợi ý:** sử dụng `filling_values=-1`.

In [ ]:
# TODO
# arr_ex2 = np.genfromtxt(
#     SOURCE_DIR / "numeric_sales.csv",
#     delimiter=",",
#     skip_header=1,
#     filling_values=...
# )
# print(arr_ex2[:, 2])

## 4. Ghi dữ liệu số bằng `np.savetxt()`

In [ ]:
processed = np.array([
    [1, 120.25, 80.10],
    [2, 150.75, 95.00],
    [3, 180.50, 110.20]
])

np.savetxt(
    OUTPUT_DIR / "processed_numeric_sales.csv",
    processed,
    delimiter=",",
    fmt="%.2f",
    header="Month,Revenue,Cost",
    comments=""
)

print((OUTPUT_DIR / "processed_numeric_sales.csv").read_text())

### Bài tập 3 — `np.savetxt()`

Lưu `sales_with_missing` vào `numeric_sales_filled.csv`.

Yêu cầu:

- các giá trị được phân cách bằng dấu phẩy;
- mỗi số có hai chữ số sau dấu thập phân;
- tiêu đề là `Month,Revenue,Cost`;
- không thêm ký tự `#` trước dòng tiêu đề.

In [ ]:
# TODO
# np.savetxt(
#     OUTPUT_DIR / "numeric_sales_filled.csv",
#     sales_with_missing,
#     delimiter=...,
#     fmt=...,
#     header=...,
#     comments=...
# )

# Phần 2. Làm việc với CSV bằng Pandas

## 5. Đọc CSV cơ bản bằng `pd.read_csv()`

In [ ]:
customers_raw = pd.read_csv(SOURCE_DIR / "customers.csv")
print(customers_raw.head().to_string(index=False))
print("\nShape:", customers_raw.shape)

### Bài tập 4 — Đọc CSV cơ bản

Đọc `customers.csv` vào DataFrame có tên `customers_ex4`.

Hiển thị:

1. năm dòng đầu tiên;
2. kích thước DataFrame;
3. danh sách tên cột.

In [ ]:
# TODO
# customers_ex4 = pd.read_csv(...)
# print(customers_ex4.head())
# print(customers_ex4.shape)
# print(customers_ex4.columns)

## 6. Chỉ đọc các cột cần thiết bằng `usecols`

Lựa chọn cột ngay khi nhập dữ liệu có thể giúp giảm lượng bộ nhớ sử dụng và loại bỏ sớm các biến không cần thiết.

In [ ]:
customer_subset = pd.read_csv(
    SOURCE_DIR / "customers.csv",
    usecols=["CustomerID", "City", "TotalSpent"]
)

print(customer_subset.head().to_string(index=False))

### Bài tập 5 — `usecols`

Chỉ đọc các cột:

- `CustomerID`;
- `CustomerName`;
- `Segment`;
- `TotalSpent`.

Lưu kết quả vào `customers_ex5`.

In [ ]:
# TODO
# customers_ex5 = pd.read_csv(
#     SOURCE_DIR / "customers.csv",
#     usecols=[...]
# )

## 7. Kiểm soát kiểu dữ liệu và giá trị thiếu

Các trường mã định danh như `CustomerID` thường nên được xem là nhãn, không phải đại lượng số học.

In [ ]:
customers_typed = pd.read_csv(
    SOURCE_DIR / "customers.csv",
    dtype={"CustomerID": str, "Age": "Int64"},
    na_values=["NA", "-"]
)

print(customers_typed.dtypes)
print("\nMissing values:")
print(customers_typed.isna().sum())

### Bài tập 6 — `dtype` và `na_values`

Đọc `customers.csv` với các yêu cầu:

- `CustomerID` có kiểu `str`;
- `Age` có kiểu `Int64`;
- `NA` và `-` được xem là giá trị thiếu.

Sau đó kiểm tra `dtypes` và số lượng giá trị thiếu của từng cột.

In [ ]:
# TODO
# customers_ex6 = pd.read_csv(
#     SOURCE_DIR / "customers.csv",
#     dtype={...},
#     na_values=[...]
# )
# print(customers_ex6.dtypes)
# print(customers_ex6.isna().sum())

## 8. Chuyển cột ngày tháng bằng `parse_dates`

In [ ]:
customers_dates = pd.read_csv(
    SOURCE_DIR / "customers.csv",
    dtype={"CustomerID": str},
    parse_dates=["SignupDate"],
    na_values=["NA", "-"]
)

print(customers_dates.dtypes)
print(customers_dates[["CustomerID", "SignupDate"]].head().to_string(index=False))

### Bài tập 7 — `parse_dates`

Đọc `customers.csv` và chuyển `SignupDate` sang kiểu datetime trong quá trình nhập dữ liệu.

Sau đó in ngày đăng ký sớm nhất và muộn nhất.

In [ ]:
# TODO
# customers_ex7 = pd.read_csv(
#     SOURCE_DIR / "customers.csv",
#     dtype={"CustomerID": str},
#     parse_dates=[...],
#     na_values=["NA", "-"]
# )
# print("Earliest:", customers_ex7["SignupDate"].min())
# print("Latest:", customers_ex7["SignupDate"].max())

## 9. Kiểm tra dữ liệu sau khi nhập

Một trình tự kiểm tra hữu ích là:

```python
df.head()
df.shape
df.info()
df.dtypes
df.isna().sum()
```

In [ ]:
customers_clean = pd.read_csv(
    SOURCE_DIR / "customers.csv",
    dtype={"CustomerID": str, "Age": "Int64"},
    parse_dates=["SignupDate"],
    na_values=["NA", "-"]
)

print(customers_clean.head().to_string(index=False))
print("\nShape:", customers_clean.shape)
print("\nMissing values:")
print(customers_clean.isna().sum())

### Bài tập 8 — Kiểm tra dữ liệu

Sử dụng `customers_clean` để xác định:

1. số hàng;
2. số cột;
3. số giá trị thiếu trong `Age`;
4. số giá trị thiếu trong `Segment`.

In [ ]:
# TODO
# print("Rows:", ...)
# print("Columns:", ...)
# print("Missing Age:", ...)
# print("Missing Segment:", ...)

## 10. Xử lý tệp CSV lớn bằng `chunksize`

`chunksize` trả về một iterator gồm nhiều DataFrame nhỏ, nhờ đó dữ liệu có thể được xử lý lần lượt theo từng khối.

In [ ]:
total_amount = 0.0
row_count = 0

for chunk in pd.read_csv(
    SOURCE_DIR / "large_transactions.csv",
    chunksize=5000
):
    total_amount += chunk["Amount"].sum()
    row_count += len(chunk)

print("Số dòng đã xử lý:", row_count)
print("Tổng giá trị:", round(total_amount, 2))

### Bài tập 9 — `chunksize`

Xử lý `large_transactions.csv` theo từng khối 4.000 dòng và tính:

1. tổng `Amount`;
2. tổng số dòng;
3. giá trị giao dịch trung bình.

Không đọc toàn bộ tệp vào một DataFrame duy nhất để thực hiện phép tính.

In [ ]:
# TODO
# total = 0.0
# n = 0
# for chunk in pd.read_csv(
#     SOURCE_DIR / "large_transactions.csv",
#     chunksize=...
# ):
#     total += ...
#     n += ...
# average = total / n
# print(total, n, average)

## 11. Ghi dữ liệu ra CSV

In [ ]:
customers_clean.to_csv(
    OUTPUT_DIR / "customers_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Đã lưu:", OUTPUT_DIR / "customers_clean.csv")

### Kiểm tra kiến thức — CSV

**Câu 1.** Tham số nào cho phép chỉ đọc các cột được lựa chọn?

A. `usecols`  
B. `columns_only`  
C. `keepcols`  
D. `filter_cols`

**Câu 2.** Tham số nào hữu ích trực tiếp khi tệp CSV lớn hơn dung lượng RAM khả dụng?

A. `chunksize`  
B. `index_col`  
C. `header`  
D. `names`

**Câu 3.** Vì sao `CustomerID` thường nên được lưu dưới dạng `str`?

A. Vì đây là mã định danh, không phải đại lượng số học.  
B. Vì cần tính giá trị trung bình của mã khách hàng.  
C. Vì mã khách hàng luôn bị thiếu.  
D. Vì Pandas không hỗ trợ mã định danh dạng số.

# Phần 3. Làm việc với Microsoft Excel

## 12. Đọc một trang tính

In [ ]:
products_df = pd.read_excel(
    SOURCE_DIR / "products.xlsx",
    sheet_name="Products",
    engine="openpyxl"
)

print(products_df.to_string(index=False))

### Bài tập 10 — `read_excel()`

Đọc trang tính `Inventory` vào `inventory_ex10` và hiển thị năm dòng đầu tiên.

In [ ]:
# TODO
# inventory_ex10 = pd.read_excel(
#     SOURCE_DIR / "products.xlsx",
#     sheet_name=...,
#     engine="openpyxl"
# )
# print(inventory_ex10.head())

## 13. Đọc toàn bộ các trang tính

Sử dụng `sheet_name=None` sẽ trả về một dictionary gồm các DataFrame.

In [ ]:
workbook = pd.read_excel(
    SOURCE_DIR / "products.xlsx",
    sheet_name=None,
    engine="openpyxl"
)

print("Các trang tính:", list(workbook.keys()))

for sheet_name, sheet_df in workbook.items():
    print(sheet_name, sheet_df.shape)

### Bài tập 11 — Làm việc với nhiều trang tính

Sử dụng `workbook` để:

1. truy cập `Products`;
2. truy cập `Inventory`;
3. kết hợp hai DataFrame theo `ProductID`.

In [ ]:
# TODO
# products_ex11 = workbook[...]
# inventory_ex11 = workbook[...]
# product_inventory = products_ex11.merge(...)
# print(product_inventory)

## 14. Ghi nhiều trang tính

In [ ]:
inventory_df = workbook["Inventory"]
product_inventory = products_df.merge(
    inventory_df,
    on="ProductID",
    how="left"
)

with pd.ExcelWriter(
    OUTPUT_DIR / "product_report.xlsx",
    engine="openpyxl"
) as writer:
    products_df.to_excel(writer, sheet_name="Products", index=False)
    inventory_df.to_excel(writer, sheet_name="Inventory", index=False)
    product_inventory.to_excel(writer, sheet_name="Combined", index=False)

print("Đã lưu:", OUTPUT_DIR / "product_report.xlsx")

### Bài tập 12 — Xuất dữ liệu Excel

Tạo `inventory_report.xlsx` gồm hai trang tính:

- `Inventory`;
- `Low_Stock`.

Quy ước một mặt hàng có tồn kho thấp khi:

```text
Stock <= ReorderLevel
```

In [ ]:
# TODO
# low_stock = inventory_df[
#     inventory_df["Stock"] <= inventory_df["ReorderLevel"]
# ]
#
# with pd.ExcelWriter(
#     OUTPUT_DIR / "inventory_report.xlsx",
#     engine="openpyxl"
# ) as writer:
#     ...

# Phần 4. Làm việc với JSON

## 15. Đọc dữ liệu JSON

In [ ]:
with open(
    SOURCE_DIR / "orders.json",
    "r",
    encoding="utf-8"
) as f:
    orders_data = json.load(f)

print("Số đơn hàng:", len(orders_data))
print(json.dumps(orders_data[0], indent=2))

### Bài tập 13 — Kiểm tra cấu trúc JSON

Với đơn hàng đầu tiên:

1. in `OrderID`;
2. in mã khách hàng;
3. in số lượng mặt hàng trong đơn.

In [ ]:
# TODO
# first_order = orders_data[0]
# print(first_order[...])
# print(first_order["Customer"][...])
# print(len(first_order[...]))

## 16. Làm phẳng JSON lồng nhau bằng `pd.json_normalize()`

Mỗi phần tử trong danh sách `Items` lồng nhau sẽ được chuyển thành một dòng riêng.

In [ ]:
order_items = pd.json_normalize(
    orders_data,
    record_path=["Items"],
    meta=[
        "OrderID",
        "OrderDate",
        ["Customer", "CustomerID"],
        ["Customer", "City"]
    ]
)

order_items = order_items.rename(columns={
    "Customer.CustomerID": "CustomerID",
    "Customer.City": "City"
})

print(order_items.to_string(index=False))

### Bài tập 14 — `pd.json_normalize()`

Tạo `items_ex14` với:

- mỗi mặt hàng tương ứng một dòng;
- `OrderID`;
- `OrderDate`;
- `CustomerID`;
- `City`;
- `ProductID`;
- `Quantity`.

Sau đó kiểm tra số dòng của DataFrame có bằng tổng số mặt hàng của tất cả đơn hàng hay không.

In [ ]:
# TODO
# items_ex14 = pd.json_normalize(
#     orders_data,
#     record_path=[...],
#     meta=[...]
# )
# total_items = sum(len(order["Items"]) for order in orders_data)
# print("Rows:", len(items_ex14))
# print("Expected:", total_items)

### Kiểm tra kiến thức — JSON

**Câu 1.** `record_path` xác định thành phần nào?

A. Danh sách lồng nhau cần mở rộng thành các dòng  
B. Tên tệp đầu ra  
C. Tên cơ sở dữ liệu  
D. Ký hiệu biểu diễn giá trị thiếu

**Câu 2.** `meta` dùng để giữ lại thông tin nào?

A. Thông tin ở cấp cha  
B. Chỉ các trường số  
C. Quyền truy cập tệp  
D. Các thẻ HTML

# Phần 5. Làm việc với HTML và PDF

## 17. Đọc bảng HTML bằng `pd.read_html()`

In [ ]:
html_tables = pd.read_html(SOURCE_DIR / "market_data.html")

print("Số bảng HTML:", len(html_tables))

for i, table in enumerate(html_tables):
    print(f"\nTable {i}")
    print(table.to_string(index=False))

### Bài tập 15 — Bảng HTML

Sử dụng `html_tables` để:

1. lưu bảng đầu tiên vào `market_prices_ex15`;
2. lưu bảng thứ hai vào `fx_rates_ex15`;
3. in kích thước của hai bảng;
4. xuất hai bảng vào hai trang tính riêng trong `market_tables.xlsx`.

In [ ]:
# TODO
# market_prices_ex15 = ...
# fx_rates_ex15 = ...
#
# with pd.ExcelWriter(
#     OUTPUT_DIR / "market_tables.xlsx",
#     engine="openpyxl"
# ) as writer:
#     ...

## 18. Trích xuất bảng từ PDF

PDF chủ yếu là định dạng trình bày. Một bảng nhìn có vẻ rõ ràng trên trang PDF chưa chắc được lưu bên trong dưới dạng cấu trúc bảng tường minh.

Ô tiếp theo tạo một báo cáo PDF nhỏ nếu môi trường đã cài `reportlab`.

In [ ]:
pdf_path = SOURCE_DIR / "sales_report.pdf"

try:
    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4
    from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer
    from reportlab.lib.styles import getSampleStyleSheet

    doc = SimpleDocTemplate(str(pdf_path), pagesize=A4)
    styles = getSampleStyleSheet()

    pdf_data = [
        ["Region", "Revenue", "Orders"],
        ["Hanoi", "4200", "38"],
        ["Danang", "2750", "24"],
        ["HCMC", "5100", "44"]
    ]

    table = Table(pdf_data)
    table.setStyle(TableStyle([
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
        ("ALIGN", (1, 1), (-1, -1), "RIGHT")
    ]))

    doc.build([
        Paragraph("Regional Sales Report", styles["Title"]),
        Spacer(1, 12),
        table
    ])

    print("Đã tạo:", pdf_path)

except ImportError:
    print("Chưa cài reportlab nên bỏ qua bước tạo PDF.")

In [ ]:
try:
    import pdfplumber

    if pdf_path.exists():
        with pdfplumber.open(pdf_path) as pdf:
            first_page = pdf.pages[0]
            extracted_tables = first_page.extract_tables()

        print("Số bảng trích xuất:", len(extracted_tables))

        if extracted_tables:
            raw_table = extracted_tables[0]
            pdf_df = pd.DataFrame(raw_table[1:], columns=raw_table[0])
            print(pdf_df.to_string(index=False))

except ImportError:
    print("Chưa cài pdfplumber. Có thể cài bằng: pip install pdfplumber")

### Bài tập 16 — Kiểm tra dữ liệu trích xuất từ PDF

Nếu `pdf_df` đã được tạo:

1. kiểm tra `head()`, `shape`, `columns`, giá trị thiếu và kiểu dữ liệu;
2. chuyển `Revenue` và `Orders` sang kiểu số.

In [ ]:
# TODO
# if "pdf_df" in globals():
#     print(pdf_df.head())
#     print(pdf_df.shape)
#     print(pdf_df.columns)
#     print(pdf_df.isna().sum())
#     print(pdf_df.dtypes)
#
#     pdf_df["Revenue"] = pd.to_numeric(pdf_df["Revenue"], errors="coerce")
#     pdf_df["Orders"] = pd.to_numeric(pdf_df["Orders"], errors="coerce")

# Phần 6. Làm việc với SQLite

## 19. Kết nối đến SQLite

In [ ]:
conn = sqlite3.connect(SOURCE_DIR / "company.db")
print("Đã mở kết nối.")

## 20. Đọc kết quả SQL vào Pandas

In [ ]:
employee_df = pd.read_sql_query(
    "SELECT EmployeeID, EmployeeName, Department, Salary FROM employees",
    conn
)

print(employee_df.to_string(index=False))

### Bài tập 17 — Lọc dữ liệu bằng SQL

Viết truy vấn chỉ lấy các nhân viên thuộc phòng `Sales`.

Đọc kết quả vào DataFrame `sales_employees`.

In [ ]:
# TODO
# sales_employees = pd.read_sql_query(
#     "SELECT ... FROM employees WHERE ...",
#     conn
# )
# print(sales_employees)

## 21. Truy vấn có tham số

Truy vấn có tham số giúp tách cấu trúc câu lệnh SQL khỏi các giá trị được truyền vào.

In [ ]:
salary_threshold = 30000000

high_salary = pd.read_sql_query(
    "SELECT EmployeeID, EmployeeName, Department, Salary FROM employees WHERE Salary >= ?",
    conn,
    params=(salary_threshold,)
)

print(high_salary.to_string(index=False))

### Bài tập 18 — SQL có tham số

Khai báo:

```python
department_name = "Operations"
```

Sử dụng truy vấn có tham số để lấy các nhân viên thuộc phòng này.

In [ ]:
# TODO
# department_name = "Operations"
# operations_staff = pd.read_sql_query(
#     "SELECT ... FROM employees WHERE Department = ?",
#     conn,
#     params=(department_name,)
# )
# print(operations_staff)

## 22. Ghi DataFrame vào SQLite

In [ ]:
city_summary = (
    customers_clean
    .groupby("City", as_index=False)
    .agg(
        Customers=("CustomerID", "nunique"),
        TotalSpent=("TotalSpent", "sum")
    )
)

city_summary.to_sql(
    "city_customer_summary",
    conn,
    if_exists="replace",
    index=False
)

print(
    pd.read_sql_query(
        "SELECT * FROM city_customer_summary",
        conn
    ).to_string(index=False)
)

### Bài tập 19 — `to_sql()`

Tạo bảng `segment_customer_summary` gồm:

- Segment;
- số lượng khách hàng;
- tổng chi tiêu.

Sử dụng `if_exists="replace"`.

In [ ]:
# TODO
# segment_summary = (
#     customers_clean
#     .groupby("Segment", as_index=False)
#     .agg(
#         Customers=("CustomerID", "nunique"),
#         TotalSpent=("TotalSpent", "sum")
#     )
# )
#
# segment_summary.to_sql(
#     "segment_customer_summary",
#     conn,
#     if_exists="replace",
#     index=False
# )

### Kiểm tra kiến thức — SQLite

**Câu 1.** Hàm nào mở kết nối đến SQLite?

A. `sqlite3.connect()`  
B. `pd.connect_sqlite()`  
C. `sqlite.open()`  
D. `pd.read_db()`

**Câu 2.** Hàm Pandas nào đọc trực tiếp kết quả truy vấn SQL vào DataFrame?

A. `pd.read_sql_query()`  
B. `pd.sql_to_frame()`  
C. `pd.read_database_file()`  
D. `pd.load_sql()`

**Câu 3.** Vì sao nên sử dụng truy vấn có tham số thay vì nối chuỗi trực tiếp?

A. Vì cách này tách cấu trúc SQL khỏi giá trị được truyền vào và giúp giảm nguy cơ SQL Injection.  
B. Vì cách này tự động loại bỏ mọi giá trị thiếu.  
C. Vì không cần mở kết nối cơ sở dữ liệu.  
D. Vì hệ thống tự động tạo chỉ mục.

# Phần 7. Dữ liệu dạng Document của MongoDB

## 23. Dữ liệu hướng document

Một collection trong MongoDB chứa các document thay vì các dòng theo mô hình quan hệ. Document có thể chứa dictionary và list lồng nhau.

Ví dụ đầu tiên sử dụng dictionary Python thông thường để notebook vẫn có thể chạy ngay cả khi máy không có MongoDB server.

In [ ]:
mongo_style_orders = [
    {
        "_id": "m001",
        "OrderID": "M001",
        "Customer": {"CustomerID": "001", "City": "Hanoi"},
        "TotalAmount": 1500,
        "Status": "PAID"
    },
    {
        "_id": "m002",
        "OrderID": "M002",
        "Customer": {"CustomerID": "003", "City": "Danang"},
        "TotalAmount": 820,
        "Status": "SHIPPED"
    },
    {
        "_id": "m003",
        "OrderID": "M003",
        "Customer": {"CustomerID": "008", "City": "HCMC"},
        "TotalAmount": 2100,
        "Status": "PAID"
    }
]

mongo_df = pd.json_normalize(mongo_style_orders)
print(mongo_df.to_string(index=False))

### Bài tập 20 — Dữ liệu dạng MongoDB

Sử dụng `mongo_style_orders` để:

1. chỉ giữ các document có `Status == "PAID"`;
2. chuyển chúng thành DataFrame;
3. tính tổng `TotalAmount`.

In [ ]:
# TODO
# paid_docs = [
#     doc for doc in mongo_style_orders
#     if doc["Status"] == "PAID"
# ]
# paid_df = pd.json_normalize(...)
# print(...)

## 24. Kết nối MongoDB thực tế — Phần mở rộng

Phần này không bắt buộc. Code chỉ chạy đầy đủ khi:

- đã cài `pymongo`;
- MongoDB server đang hoạt động tại `mongodb://localhost:27017/`.

Thời gian chờ được đặt ngắn để notebook không bị treo khi không có MongoDB server.

In [ ]:
try:
    from pymongo import MongoClient

    client = MongoClient(
        "mongodb://localhost:27017/",
        serverSelectionTimeoutMS=1000
    )

    client.admin.command("ping")

    db = client["retail_lab"]
    collection = db["orders"]

    collection.delete_many({})
    collection.insert_many(mongo_style_orders)

    live_documents = list(collection.find({"Status": "PAID"}))
    live_df = pd.json_normalize(live_documents)

    if "_id" in live_df.columns:
        live_df["_id"] = live_df["_id"].astype(str)

    print(live_df.to_string(index=False))

except Exception as exc:
    print("Bỏ qua ví dụ MongoDB trực tiếp:", type(exc).__name__)

# Phần 8. Data Pipeline tích hợp nhiều nguồn

## 25. Bối cảnh kinh doanh

Xây dựng một tập dữ liệu phân tích đơn hàng bằng cách tích hợp:

- thông tin khách hàng từ CSV;
- thông tin sản phẩm từ Excel;
- chi tiết mặt hàng trong đơn từ JSON lồng nhau.

Bảng cuối cùng cần chứa:

- OrderID;
- OrderDate;
- CustomerID;
- CustomerName;
- City;
- ProductID;
- ProductName;
- Category;
- Quantity;
- UnitPrice;
- Revenue.

## 26. Bước 1 — Đọc và kiểm tra các nguồn dữ liệu

In [ ]:
customers_pipeline = pd.read_csv(
    SOURCE_DIR / "customers.csv",
    dtype={"CustomerID": str, "Age": "Int64"},
    parse_dates=["SignupDate"],
    na_values=["NA", "-"]
)

products_pipeline = pd.read_excel(
    SOURCE_DIR / "products.xlsx",
    sheet_name="Products"
)

with open(SOURCE_DIR / "orders.json", encoding="utf-8") as f:
    orders_pipeline = json.load(f)

items_pipeline = pd.json_normalize(
    orders_pipeline,
    record_path=["Items"],
    meta=[
        "OrderID",
        "OrderDate",
        ["Customer", "CustomerID"],
        ["Customer", "City"]
    ]
).rename(columns={
    "Customer.CustomerID": "CustomerID",
    "Customer.City": "OrderCity"
})

items_pipeline["OrderDate"] = pd.to_datetime(items_pipeline["OrderDate"])

print("Khách hàng:", customers_pipeline.shape)
print("Sản phẩm:", products_pipeline.shape)
print("Dòng mặt hàng trong đơn:", items_pipeline.shape)

### Bài tập 21 — Kiểm tra khóa trước khi tích hợp

Kiểm tra:

1. `CustomerID` có duy nhất trong bảng khách hàng hay không;
2. `ProductID` có duy nhất trong bảng sản phẩm hay không;
3. có `ProductID` nào xuất hiện trong đơn hàng nhưng không có trong bảng sản phẩm hay không.

In [ ]:
# TODO
# print("CustomerID unique:", customers_pipeline["CustomerID"].is_unique)
# print("ProductID unique:", products_pipeline["ProductID"].is_unique)
#
# missing_product_ids = (
#     set(items_pipeline["ProductID"])
#     - set(products_pipeline["ProductID"])
# )
# print("Missing Product IDs:", missing_product_ids)

## 27. Bước 2 — Tích hợp các nguồn dữ liệu

In [ ]:
order_analytics = (
    items_pipeline
    .merge(
        products_pipeline,
        on="ProductID",
        how="left",
        validate="many_to_one"
    )
    .merge(
        customers_pipeline[
            ["CustomerID", "CustomerName", "City", "Segment"]
        ],
        on="CustomerID",
        how="left",
        validate="many_to_one"
    )
)

order_analytics["Revenue"] = (
    order_analytics["Quantity"]
    * order_analytics["UnitPrice"]
)

print(order_analytics.head().to_string(index=False))
print("\nMissing values after merge:")
print(order_analytics.isna().sum())

### Bài tập 22 — Kiểm tra dữ liệu sau khi tích hợp

Sử dụng `order_analytics` để:

1. so sánh `OrderCity` và `City`;
2. đếm số dòng có hai giá trị này khác nhau;
3. kiểm tra `Revenue` không chứa giá trị thiếu;
4. tính tổng Revenue.

In [ ]:
# TODO
# city_mismatch = order_analytics["OrderCity"] != order_analytics["City"]
# print("City mismatches:", city_mismatch.sum())
# print("Missing Revenue:", order_analytics["Revenue"].isna().sum())
# print("Total Revenue:", order_analytics["Revenue"].sum())

## 28. Bước 3 — Tạo các bảng KPI

In [ ]:
city_kpi = (
    order_analytics
    .groupby("City", as_index=False)
    .agg(
        Revenue=("Revenue", "sum"),
        Orders=("OrderID", "nunique"),
        Customers=("CustomerID", "nunique")
    )
)

category_kpi = (
    order_analytics
    .groupby("Category", as_index=False)
    .agg(
        Revenue=("Revenue", "sum"),
        Quantity=("Quantity", "sum")
    )
)

print("KPI theo thành phố")
print(city_kpi.to_string(index=False))

print("\nCategory KPI")
print(category_kpi.to_string(index=False))

## 29. Bước 4 — Lưu kết quả phân tích

In [ ]:
analytics_conn = sqlite3.connect(OUTPUT_DIR / "analytics.db")

order_analytics.to_sql(
    "order_analytics",
    analytics_conn,
    if_exists="replace",
    index=False
)
city_kpi.to_sql(
    "city_kpi",
    analytics_conn,
    if_exists="replace",
    index=False
)
category_kpi.to_sql(
    "category_kpi",
    analytics_conn,
    if_exists="replace",
    index=False
)

with pd.ExcelWriter(
    OUTPUT_DIR / "analytics_dashboard.xlsx",
    engine="openpyxl"
) as writer:
    order_analytics.to_excel(writer, sheet_name="Details", index=False)
    city_kpi.to_excel(writer, sheet_name="City_KPI", index=False)
    category_kpi.to_excel(writer, sheet_name="Category_KPI", index=False)

analytics_conn.close()

print("Đã tạo:", OUTPUT_DIR / "analytics.db")
print("Đã tạo:", OUTPUT_DIR / "analytics_dashboard.xlsx")

### Bài tập 23 — Mở rộng pipeline tích hợp

Tạo `segment_kpi` gồm:

- Segment;
- số khách hàng duy nhất;
- số đơn hàng duy nhất;
- tổng Revenue.

Lưu kết quả:

1. vào `analytics.db` với tên bảng `segment_kpi`;
2. vào `segment_report.xlsx` với tên trang tính `Segment_KPI`.

In [ ]:
# TODO
# segment_kpi = (
#     order_analytics
#     .groupby(...)
#     .agg(...)
#     .reset_index()
# )
#
# conn2 = sqlite3.connect(OUTPUT_DIR / "analytics.db")
# segment_kpi.to_sql("segment_kpi", conn2, if_exists="replace", index=False)
# conn2.close()
#
# with pd.ExcelWriter(
#     OUTPUT_DIR / "segment_report.xlsx",
#     engine="openpyxl"
# ) as writer:
#     segment_kpi.to_excel(writer, sheet_name="Segment_KPI", index=False)

# Phần 9. Các nguyên tắc thực hành tốt

## 30. Quy trình kiểm tra có thể tái sử dụng

Một quy trình thực hành phù hợp là:

```text
Nguồn dữ liệu
      ↓
Đọc dữ liệu
      ↓
Kiểm tra
      ↓
Xác thực
      ↓
Chuyển đổi
      ↓
Tích hợp
      ↓
Phân tích
      ↓
Lưu trữ
```

Các kiểm tra hữu ích gồm:

- `head()`;
- `shape`;
- `info()`;
- `dtypes`;
- `isna().sum()`;
- tính duy nhất của khóa;
- các khóa không tìm thấy bản ghi tương ứng khi merge;
- kiểm tra miền giá trị và các nhóm phân loại.

In [ ]:
print("Số dòng:", len(order_analytics))
print("Số cột:", len(order_analytics.columns))
print("Số đơn hàng duy nhất:", order_analytics["OrderID"].nunique())
print("Số khách hàng duy nhất:", order_analytics["CustomerID"].nunique())
print("Số ProductName bị thiếu:", order_analytics["ProductName"].isna().sum())
print("Số CustomerName bị thiếu:", order_analytics["CustomerName"].isna().sum())

### Bài tập 24 — Lựa chọn định dạng lưu trữ

Chọn định dạng phù hợp cho từng tình huống và giải thích lựa chọn.

1. Gửi một bảng dữ liệu đơn giản cho đồng nghiệp.
2. Tạo báo cáo quản trị gồm nhiều trang tính.
3. Trao đổi dữ liệu lồng nhau qua Web API.
4. Lưu dữ liệu giao dịch cần hỗ trợ truy vấn SQL.
5. Lưu các document có cấu trúc linh hoạt và lồng nhau.
6. Lưu một bảng dữ liệu phân tích lớn để tái sử dụng hiệu quả.

Các lựa chọn có thể gồm CSV, Excel, JSON, SQLite, MongoDB và Parquet.

# Phần 10. Bài thực hành mẫu có lời giải

## 31. Tổng hợp chi tiêu khách hàng

### Yêu cầu

Sử dụng `customers.csv` để:

1. đọc ID dưới dạng chuỗi;
2. chuyển `SignupDate` sang datetime;
3. xem `NA` và `-` là giá trị thiếu;
4. tính tổng và mức chi tiêu trung bình theo City;
5. lưu kết quả vào CSV và SQLite.

Ô tiếp theo cung cấp một lời giải tham khảo hoàn chỉnh.

In [ ]:
customers_worked = pd.read_csv(
    SOURCE_DIR / "customers.csv",
    dtype={"CustomerID": str, "Age": "Int64"},
    parse_dates=["SignupDate"],
    na_values=["NA", "-"]
)

customer_city_summary = (
    customers_worked
    .groupby("City", as_index=False)
    .agg(
        Customers=("CustomerID", "nunique"),
        TotalSpent=("TotalSpent", "sum"),
        AverageSpent=("TotalSpent", "mean")
    )
)

customer_city_summary["AverageSpent"] = (
    customer_city_summary["AverageSpent"].round(2)
)

customer_city_summary.to_csv(
    OUTPUT_DIR / "customer_city_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

conn_ref = sqlite3.connect(OUTPUT_DIR / "customer_summary.db")
customer_city_summary.to_sql(
    "customer_city_summary",
    conn_ref,
    if_exists="replace",
    index=False
)
conn_ref.close()

print(customer_city_summary.to_string(index=False))

# Phần 11. Project thực hành tự làm

## 32. Project Truy xuất và Lưu trữ Dữ liệu Bán lẻ

Bạn là chuyên viên phân tích dữ liệu đang chuẩn bị một tập dữ liệu phân tích có thể tái sử dụng cho bộ phận quản lý.

### Các nguồn bắt buộc

Sử dụng:

- `customers.csv`;
- `products.xlsx`;
- `orders.json`;
- `company.db`.

### Yêu cầu thực hiện

1. Đọc tất cả nguồn dữ liệu liên quan bằng các tùy chọn phù hợp.
2. Kiểm tra kích thước, kiểu dữ liệu, giá trị thiếu và tính duy nhất của khóa.
3. Làm phẳng dữ liệu đơn hàng JSON lồng nhau.
4. Tích hợp khách hàng, sản phẩm và các dòng đơn hàng.
5. Tính Revenue cho từng dòng đơn hàng.
6. Tạo ít nhất ba bảng KPI.
7. Thực hiện ít nhất một truy vấn SQLite có tham số.
8. Lưu bảng dữ liệu phân tích chi tiết vào SQLite.
9. Xuất các bảng KPI vào một workbook Excel nhiều trang tính.
10. Xuất ít nhất một bảng cuối cùng sang CSV bằng mã hóa UTF-8 phù hợp với Excel.

### Yêu cầu xác thực dữ liệu

Tập dữ liệu cuối cùng cần thỏa mãn:

- `CustomerID` vẫn là chuỗi;
- `OrderDate` là datetime trước khi xuất;
- mọi mặt hàng trong đơn đều có sản phẩm hợp lệ;
- mọi mặt hàng trong đơn đều có khách hàng hợp lệ;
- `Revenue = Quantity × UnitPrice`;
- các trường phân tích cốt lõi không chứa giá trị thiếu.

In [ ]:
# TODO — KHU VỰC LÀM PROJECT CUỐI

# 1. Đọc các nguồn dữ liệu
# ...

# 2. Kiểm tra và xác thực
# ...

# 3. Làm phẳng JSON
# ...

# 4. Tích hợp dữ liệu
# ...

# 5. Tạo biến Revenue
# ...

# 6. Xây dựng các bảng KPI
# ...

# 7. Truy vấn SQLite
# ...

# 8. Lưu kết quả
# ...

print("Hoàn thành project cuối trong ô này hoặc thêm các ô mới bên dưới.")

## 33. Tự kiểm tra bằng assertion — Phần mở rộng

Sau khi hoàn thành project, điều chỉnh các câu lệnh assertion sau theo tên biến của bạn:

```python
assert final_df["CustomerID"].dtype == object
assert pd.api.types.is_datetime64_any_dtype(final_df["OrderDate"])
assert final_df["CustomerName"].notna().all()
assert final_df["ProductName"].notna().all()
assert final_df["Revenue"].notna().all()

assert np.allclose(
    final_df["Revenue"],
    final_df["Quantity"] * final_df["UnitPrice"]
)
```

Assertion giúp chuyển các giả định quan trọng về chất lượng dữ liệu thành các phép kiểm tra có thể thực thi.

## 34. Bảng tra cứu nhanh — Data Access và Storage

| Mục tiêu | Câu lệnh chính |
|---|---|
| Đọc tệp văn bản số đơn giản | `np.loadtxt(...)` |
| Đọc dữ liệu số có giá trị thiếu | `np.genfromtxt(...)` |
| Ghi dữ liệu NumPy ra tệp văn bản | `np.savetxt(...)` |
| Đọc CSV | `pd.read_csv(...)` |
| Chỉ đọc các cột được chọn | `usecols=[...]` |
| Quy định kiểu dữ liệu | `dtype={...}` |
| Chuyển cột ngày tháng | `parse_dates=[...]` |
| Quy định ký hiệu giá trị thiếu | `na_values=[...]` |
| Đọc CSV lớn theo từng khối | `chunksize=...` |
| Ghi CSV | `df.to_csv(...)` |
| Đọc Excel | `pd.read_excel(...)` |
| Đọc toàn bộ trang tính Excel | `sheet_name=None` |
| Ghi nhiều trang tính | `pd.ExcelWriter(...)` |
| Đọc JSON | `json.load(...)`, `pd.read_json(...)` |
| Làm phẳng JSON lồng nhau | `pd.json_normalize(...)` |
| Đọc bảng HTML | `pd.read_html(...)` |
| Trích xuất bảng PDF | `pdfplumber` |
| Kết nối SQLite | `sqlite3.connect(...)` |
| Đọc SQL vào Pandas | `pd.read_sql_query(...)` |
| Ghi Pandas vào SQL | `df.to_sql(...)` |
| Chuẩn hóa document MongoDB | `pd.json_normalize(...)` |

### Checklist trước khi nộp

- [ ] Đã sử dụng các tùy chọn nhập dữ liệu phù hợp.
- [ ] Các trường ID có kiểu dữ liệu phù hợp.
- [ ] Các cột ngày tháng đã được chuyển đúng kiểu.
- [ ] Đã kiểm tra giá trị thiếu.
- [ ] Đã xác thực các khóa dùng để merge.
- [ ] Đã kiểm tra giá trị thiếu sau khi merge.
- [ ] Dữ liệu lớn được xử lý theo cách phù hợp.
- [ ] Truy vấn SQL sử dụng tham số khi giá trị đến từ biến.
- [ ] Định dạng đầu ra phù hợp với mục đích sử dụng.
- [ ] Pipeline có thể chạy lại từ đầu.

## Kết luận

Notebook đã đi qua toàn bộ quy trình:

```text
Tạo / Thu thập nguồn dữ liệu
            ↓
       Đọc dữ liệu
            ↓
        Kiểm tra
            ↓
        Xác thực
            ↓
       Chuyển đổi
            ↓
        Tích hợp
            ↓
        Phân tích
            ↓
      Lưu trữ / Báo cáo
```

Bước tiếp theo là thay các nguồn dữ liệu mô phỏng bằng dữ liệu thực tế nhưng vẫn giữ nguyên kỷ luật về truy xuất, kiểm tra, tích hợp và lưu trữ dữ liệu.

In [ ]:
try:
    conn.close()
except Exception:
    pass

print("Đã hoàn thành phần thực hành cốt lõi.")
print("Kết quả được lưu tại:", OUTPUT_DIR.resolve())